In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

In [ ]:
def flatten_columns(df):
    df.columns = [
        "_".join(col).upper() if isinstance(col, tuple) else col
        for col in df.columns
    ]
    return df

## 1. Загрузка исходных данных

Чтение таблиц датасета **Home Credit Default Risk** из директории `data/raw/`.
Загружаются следующие файлы:

- `application_train.csv` — основная таблица с характеристиками заявок
- `bureau.csv` — данные кредитного бюро по предыдущим кредитам клиентов
- `previous_application.csv` — история предыдущих заявок в Home Credit
- `POS_CASH_balance.csv` — ежемесячные остатки по POS-кредитам
- `installments_payments.csv` — история платежей по предыдущим кредитам
- `credit_card_balance.csv` — ежемесячный баланс кредитных карт


In [ ]:
import os

DATA_PATH = "data/raw"       # сырые CSV файлы
PROCESSED_PATH = "data/processed"  # обработанные данные
FEATURES_PATH = "features"   # описания признаков

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(FEATURES_PATH, exist_ok=True)

In [ ]:
np.random.seed(91)

In [ ]:
app = pd.read_csv(DATA_PATH + "/application_train.csv")

print("Application shape:", app.shape)

Application shape: (307511, 122)


## 2. Агрегация вспомогательных таблиц

Для каждого клиента (`SK_ID_CURR`) вычисляются агрегированные показатели по всем вспомогательным таблицам. Используются статистики: среднее, максимум, минимум, сумма — в зависимости от экономического смысла признака. Все таблицы объединяются с основной через `LEFT JOIN` по `SK_ID_CURR`.


In [ ]:
bureau = pd.read_csv(DATA_PATH + "/bureau.csv")

bureau_agg = bureau.groupby("SK_ID_CURR").agg({
    "DAYS_CREDIT": ["mean", "min", "max"],
    "AMT_CREDIT_SUM": ["mean", "sum"],
    "AMT_CREDIT_SUM_DEBT": ["mean", "sum"],
    "AMT_CREDIT_SUM_OVERDUE": ["sum"],
    "CREDIT_DAY_OVERDUE": ["max"],
})

bureau_agg = flatten_columns(bureau_agg).reset_index()

print("bureau_agg:", bureau_agg.shape)

bureau_agg: (305811, 10)


In [ ]:
prev = pd.read_csv(DATA_PATH + "/previous_application.csv")

prev_agg = prev.groupby("SK_ID_CURR").agg({
    "AMT_APPLICATION": ["mean", "max"],
    "AMT_CREDIT": ["mean", "max"],
    "CNT_PAYMENT": ["mean"],
    "DAYS_DECISION": ["mean"],
})

prev_agg = flatten_columns(prev_agg).reset_index()

print("prev_agg:", prev_agg.shape)

prev_agg: (338857, 7)


In [ ]:
pos = pd.read_csv(DATA_PATH + "/POS_CASH_balance.csv")

pos_agg = pos.groupby("SK_ID_CURR").agg({
    "MONTHS_BALANCE": ["mean", "min"],
    "CNT_INSTALMENT": ["mean"],
    "CNT_INSTALMENT_FUTURE": ["mean"],
    "SK_DPD": ["max"],
})

pos_agg = flatten_columns(pos_agg).reset_index()

print("pos_agg:", pos_agg.shape)

pos_agg: (337252, 6)


In [ ]:
inst = pd.read_csv(DATA_PATH + "/installments_payments.csv")

inst_agg = inst.groupby("SK_ID_CURR").agg({
    "AMT_PAYMENT": ["mean", "sum"],
    "AMT_INSTALMENT": ["mean"],
    "DAYS_ENTRY_PAYMENT": ["mean"],
    "DAYS_INSTALMENT": ["mean"],
})

inst_agg = flatten_columns(inst_agg).reset_index()

print("inst_agg:", inst_agg.shape)

inst_agg: (339587, 6)


In [ ]:
credit = pd.read_csv(DATA_PATH + "/credit_card_balance.csv")

credit_agg = credit.groupby("SK_ID_CURR").agg({
    "AMT_BALANCE": ["mean", "max"],
    "AMT_DRAWINGS_CURRENT": ["mean"],
    "AMT_CREDIT_LIMIT_ACTUAL": ["mean"],
    "SK_DPD": ["max"],
})

credit_agg = flatten_columns(credit_agg).reset_index()

print("credit_agg:", credit_agg.shape)

credit_agg: (103558, 6)


In [ ]:
df = app.copy()

df = df.merge(bureau_agg, on="SK_ID_CURR", how="left")
df = df.merge(prev_agg, on="SK_ID_CURR", how="left")
df = df.merge(pos_agg, on="SK_ID_CURR", how="left")
df = df.merge(inst_agg, on="SK_ID_CURR", how="left")
df = df.merge(credit_agg, on="SK_ID_CURR", how="left")

print("Final dataset:", df.shape)

Final dataset: (307511, 152)


## 3. Предобработка датасета

Перед генерацией синтетических uplift-переменных датасет проходит предварительную обработку:

- стандартизация типов данных (приведение категориальных и числовых признаков)
- первичный анализ пропущенных значений
- приведение дат и флаговых переменных к корректным типам

> Исходный датасет без обработки пропусков сохраняется отдельно — пропуски обрабатываются на этапе построения каждой модели индивидуально.


In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)

# Создаем копию датасета, в df_sim обработаем пропуски для корректной генерации uplift-переменных, при этом сохраним пропуски в исходном df
df_sim = df.copy()

df_sim = df_sim.fillna(df_sim.median(numeric_only=True))

## 4. Анализ признаков объединённого датасета

Первичный анализ объединённого датасета: распределения признаков, доля пропущенных значений, типы переменных. Результаты используются для принятия решений об обработке и генерации новых признаков.


In [ ]:
df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,AMT_PAYMENT_MEAN,AMT_PAYMENT_SUM,AMT_INSTALMENT_MEAN,DAYS_ENTRY_PAYMENT_MEAN,DAYS_INSTALMENT_MEAN,AMT_BALANCE_MEAN,AMT_BALANCE_MAX,AMT_DRAWINGS_CURRENT_MEAN,AMT_CREDIT_LIMIT_ACTUAL_MEAN,SK_DPD_MAX_y
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,11559.247105,219625.695,11559.247105,-315.421053,-295.000000,NaN,NaN,NaN,NaN,NaN
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,64754.586000,1618864.650,64754.586000,-1385.320000,-1378.160000,NaN,NaN,NaN,NaN,NaN
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,7096.155000,21288.465,7096.155000,-761.666667,-754.000000,NaN,NaN,NaN,NaN,NaN
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,62947.088438,1007153.415,62947.088438,-271.625000,-252.250000,0.0,0.0,0.0,270000.0,0.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,12214.060227,806127.975,12666.444545,-1032.242424,-1028.606061,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,7492.924286,52450.470,7492.924286,-156.285714,-120.000000,NaN,NaN,NaN,NaN,NaN
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,10069.867500,60419.205,10069.867500,-2393.833333,-2391.000000,NaN,NaN,NaN,NaN,NaN
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,4115.915357,57622.815,4399.707857,-2387.428571,-2372.928571,NaN,NaN,NaN,NaN,NaN
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,10239.832895,194556.825,10239.832895,-161.263158,-142.263158,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 152 entries, SK_ID_CURR to SK_DPD_MAX_y
dtypes: float64(95), int64(41), object(16)
memory usage: 356.6+ MB


In [ ]:
df['TARGET'].value_counts(normalize=True).round(4) * 100

,proportion
TARGET,
0,91.93
1,8.07


In [ ]:
dt_cols = df.select_dtypes(include=['datetime64']).columns.to_list()
object_cols = df.select_dtypes(include=['object']).columns.to_list()
numeric_cols = df.select_dtypes(include=['number']).columns.to_list()
flg_cols = [col for col in numeric_cols if "FLAG_" in col]

numeric_cols = sorted(list(set(numeric_cols) - set(flg_cols)))

numeric_cols.remove('TARGET')

In [ ]:
dt_cols

[]

In [ ]:
object_cols

['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE']

In [ ]:
numeric_cols

['AMT_ANNUITY',
 'AMT_APPLICATION_MAX',
 'AMT_APPLICATION_MEAN',
 'AMT_BALANCE_MAX',
 'AMT_BALANCE_MEAN',
 'AMT_CREDIT',
 'AMT_CREDIT_LIMIT_ACTUAL_MEAN',
 'AMT_CREDIT_MAX',
 'AMT_CREDIT_MEAN',
 'AMT_CREDIT_SUM_DEBT_MEAN',
 'AMT_CREDIT_SUM_DEBT_SUM',
 'AMT_CREDIT_SUM_MEAN',
 'AMT_CREDIT_SUM_OVERDUE_SUM',
 'AMT_CREDIT_SUM_SUM',
 'AMT_DRAWINGS_CURRENT_MEAN',
 'AMT_GOODS_PRICE',
 'AMT_INCOME_TOTAL',
 'AMT_INSTALMENT_MEAN',
 'AMT_PAYMENT_MEAN',
 'AMT_PAYMENT_SUM',
 'AMT_REQ_CREDIT_BUREAU_DAY',
 'AMT_REQ_CREDIT_BUREAU_HOUR',
 'AMT_REQ_CREDIT_BUREAU_MON',
 'AMT_REQ_CREDIT_BUREAU_QRT',
 'AMT_REQ_CREDIT_BUREAU_WEEK',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'APARTMENTS_AVG',
 'APARTMENTS_MEDI',
 'APARTMENTS_MODE',
 'BASEMENTAREA_AVG',
 'BASEMENTAREA_MEDI',
 'BASEMENTAREA_MODE',
 'CNT_CHILDREN',
 'CNT_FAM_MEMBERS',
 'CNT_INSTALMENT_FUTURE_MEAN',
 'CNT_INSTALMENT_MEAN',
 'CNT_PAYMENT_MEAN',
 'COMMONAREA_AVG',
 'COMMONAREA_MEDI',
 'COMMONAREA_MODE',
 'CREDIT_DAY_OVERDUE_MAX',
 'DAYS_BIRTH',
 'DAYS_CREDIT_M

In [ ]:
flg_cols

['FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'FLAG_DOCUMENT_2',
 'FLAG_DOCUMENT_3',
 'FLAG_DOCUMENT_4',
 'FLAG_DOCUMENT_5',
 'FLAG_DOCUMENT_6',
 'FLAG_DOCUMENT_7',
 'FLAG_DOCUMENT_8',
 'FLAG_DOCUMENT_9',
 'FLAG_DOCUMENT_10',
 'FLAG_DOCUMENT_11',
 'FLAG_DOCUMENT_12',
 'FLAG_DOCUMENT_13',
 'FLAG_DOCUMENT_14',
 'FLAG_DOCUMENT_15',
 'FLAG_DOCUMENT_16',
 'FLAG_DOCUMENT_17',
 'FLAG_DOCUMENT_18',
 'FLAG_DOCUMENT_19',
 'FLAG_DOCUMENT_20',
 'FLAG_DOCUMENT_21']

## 5. Устранение выявленных аномалий

Коррекция аномалий, выявленных в ходе предварительного анализа: некорректных типов данных, технических артефактов в значениях признаков и нарушений доменных ограничений.


In [ ]:
df = df.rename(columns={
    "SK_DPD_MAX_x": "SK_DPD_MAX_POS",
    "SK_DPD_MAX_y": "SK_DPD_MAX_CC"
})


id_cols = ["SK_ID_CURR"]

In [ ]:
flg_cols = [
    col for col in df.columns
    if col.startswith("FLAG_")
] + [
    "LIVE_CITY_NOT_WORK_CITY",
    "LIVE_REGION_NOT_WORK_REGION",
    "REG_CITY_NOT_LIVE_CITY",
    "REG_CITY_NOT_WORK_CITY",
    "REG_REGION_NOT_LIVE_REGION",
    "REG_REGION_NOT_WORK_REGION"
]


target_cols = ["TARGET"]

uplift_cols = [
    "BASE_PD",
    "CONTACT_PROPENSITY",
    "COMMUNICATION",
    "RISK_SEGMENT",
    "CONTACT_HISTORY",
    "PREFERRED_CHANNEL",
    "INTERACTION_SCORE",
    "DELAY_FLAG",
    "TRUE_UPLIFT",
]


In [ ]:
exclude_cols = set(
    id_cols +
    uplift_cols +
    flg_cols +
    object_cols +
    target_cols
)

numeric_cols = [
    col for col in df.columns
    if col not in exclude_cols
]

In [ ]:
all_grouped = set(
    id_cols +
    uplift_cols +
    flg_cols +
    object_cols +
    numeric_cols +
    target_cols
)

missing_cols = set(df.columns) - all_grouped
extra_cols = all_grouped - set(df.columns)

print("Пропущенные:", missing_cols)
print("Лишние:", extra_cols)

Пропущенные: set()
Лишние: {'CONTACT_HISTORY', 'DELAY_FLAG', 'BASE_PD', 'COMMUNICATION', 'PREFERRED_CHANNEL', 'INTERACTION_SCORE', 'TRUE_UPLIFT', 'CONTACT_PROPENSITY', 'RISK_SEGMENT'}


In [ ]:
print(f"Всего {len(id_cols)} id cols, {len(uplift_cols)} uplift_cols, {len(flg_cols)} flg_cols, {len(object_cols)} object_cols, {len(numeric_cols)} numeric_cols, {len(target_cols)} target_cols")

Всего 1 id cols, 9 uplift_cols, 34 flg_cols, 16 object_cols, 102 numeric_cols, 1 target_cols


## 6. Реестр признаков датасета

Формирование структурированного описания всех переменных датасета с указанием типа данных, доли пропусков и краткого экономического смысла. Файл сохраняется в `features/all_features_description.xlsx`.


In [ ]:
base_feature_desc = {
    "DAYS_CREDIT": "Дни с момента получения кредита",
    "AMT_CREDIT_SUM": "Сумма кредита",
    "AMT_CREDIT_SUM_DEBT": "Текущая задолженность",
    "AMT_CREDIT_SUM_OVERDUE": "Просроченная сумма",
    "CREDIT_DAY_OVERDUE": "Количество дней просрочки",

    "AMT_APPLICATION": "Запрошенная сумма кредита",
    "AMT_CREDIT": "Одобренная сумма кредита",
    "CNT_PAYMENT": "Количество платежей",
    "DAYS_DECISION": "Дни с момента решения по заявке",

    "MONTHS_BALANCE": "Месяцы баланса",
    "CNT_INSTALMENT": "Количество рассрочек",
    "CNT_INSTALMENT_FUTURE": "Будущие платежи",
    "SK_DPD": "Просрочка по дням",

    "AMT_PAYMENT": "Сумма платежа",
    "AMT_INSTALMENT": "Сумма планового платежа",
    "DAYS_ENTRY_PAYMENT": "Дата фактического платежа",
    "DAYS_INSTALMENT": "Дата планового платежа",

    "AMT_BALANCE": "Баланс по кредитной карте",
    "AMT_DRAWINGS_CURRENT": "Текущие снятия",
    "AMT_CREDIT_LIMIT_ACTUAL": "Кредитный лимит",
}

In [ ]:
agg_desc = {
    "mean": "среднее значение",
    "sum": "сумма",
    "max": "максимум",
    "min": "минимум"
}

In [ ]:
def build_feature_dict(agg_dict, source_name):
    rows = []

    for feature, aggs in agg_dict.items():
        for agg in aggs:
            col_name = f"{feature}_{agg}".upper()

            desc = f"{agg_desc.get(agg, agg)} показателя '{base_feature_desc.get(feature, feature)}'"

            rows.append({
                "feature_name": col_name,
                "description_ru": desc,
                "source_table": source_name
            })

    return rows

In [ ]:
all_features = []

all_features += build_feature_dict({
    "DAYS_CREDIT": ["mean", "min", "max"],
    "AMT_CREDIT_SUM": ["mean", "sum"],
    "AMT_CREDIT_SUM_DEBT": ["mean", "sum"],
    "AMT_CREDIT_SUM_OVERDUE": ["sum"],
    "CREDIT_DAY_OVERDUE": ["max"],
}, "bureau")

all_features += build_feature_dict({
    "AMT_APPLICATION": ["mean", "max"],
    "AMT_CREDIT": ["mean", "max"],
    "CNT_PAYMENT": ["mean"],
    "DAYS_DECISION": ["mean"],
}, "previous_application")

all_features += build_feature_dict({
    "MONTHS_BALANCE": ["mean", "min"],
    "CNT_INSTALMENT": ["mean"],
    "CNT_INSTALMENT_FUTURE": ["mean"],
    "SK_DPD": ["max"],
}, "POS_CASH_balance")


all_features += build_feature_dict({
    "AMT_PAYMENT": ["mean", "sum"],
    "AMT_INSTALMENT": ["mean"],
    "DAYS_ENTRY_PAYMENT": ["mean"],
    "DAYS_INSTALMENT": ["mean"],
}, "installments_payments")


all_features += build_feature_dict({
    "AMT_BALANCE": ["mean", "max"],
    "AMT_DRAWINGS_CURRENT": ["mean"],
    "AMT_CREDIT_LIMIT_ACTUAL": ["mean"],
    "SK_DPD": ["max"],
}, "credit_card_balance")

In [ ]:
app_features = [
    ("AMT_INCOME_TOTAL", "Общий доход клиента"),
    ("AMT_CREDIT", "Сумма текущего кредита"),
    ("AMT_ANNUITY", "Аннуитетный платеж"),
    ("EXT_SOURCE_1", "Внешний скоринговый источник 1"),
    ("EXT_SOURCE_2", "Внешний скоринговый источник 2"),
    ("EXT_SOURCE_3", "Внешний скоринговый источник 3"),
    ("DAYS_BIRTH", "Возраст клиента (в днях)"),
    ("DAYS_EMPLOYED", "Стаж работы (в днях)"),
]

for feat, desc in app_features:
    all_features.append({
        "feature_name": feat,
        "description_ru": desc,
        "source_table": "application"
    })

In [ ]:
uplift_features = [
    ("BASE_PD", "Базовая вероятность дефолта без воздействия"),

    ("CONTACT_PROPENSITY",
     "Вероятность назначения коммуникации (propensity score)"),

    ("COMMUNICATION",
     "Фактический канал коммуникации с клиентом"),

    ("RISK_SEGMENT",
     "Сегмент риска клиента (low/medium/high)"),

    ("CONTACT_HISTORY",
     "Количество предыдущих контактов с клиентом"),

    ("PREFERRED_CHANNEL",
     "Предпочтительный канал коммуникации клиента"),

    ("INTERACTION_SCORE",
     "Индекс взаимодействия признаков (доход и кредитная нагрузка)"),

    ("DELAY_FLAG",
     "Флаг отложенного эффекта коммуникации"),

    ("TRUE_UPLIFT",
     "Истинный эффект коммуникации на вероятность дефолта"),
]

In [ ]:
for feat, desc in uplift_features:
    all_features.append({
        "feature_name": feat,
        "description_ru": desc,
        "source_table": "uplift_simulation"
    })

In [ ]:
feature_df = pd.DataFrame(all_features)

def get_feature_type(col):
    if col in id_cols:
        return "id"
    elif col in target_cols:
        return "target"
    elif col in uplift_cols:
        return "uplift"
    elif col in flg_cols:
        return "binary"
    elif col in object_cols:
        return "categorical"
    else:
        return "numeric"

feature_df["feature_type"] = feature_df["feature_name"].apply(get_feature_type)

In [ ]:
feature_df.to_excel(os.path.join(FEATURES_PATH, "all_features_description.xlsx"), index=False)

---

## 7. Синтетическая генерация uplift-переменных

Поскольку исходный датасет Home Credit не содержит информации о коммуникациях с клиентами и их причинном эффекте, для решения задачи uplift-моделирования разработан алгоритм синтетической генерации релевантных переменных. Методология имитирует реальные CRM-процессы банка с учётом selection bias, гетерогенности treatment effect и communication fatigue.


### 7.1 Типы коммуникационных воздействий

В модели используются четыре типа воздействий (`COMMUNICATION`), соответствующих практике контакт-центров:


| Значение | Описание |
|---|---|
| `none` | Отсутствие коммуникации (контрольная группа) |
| `sms` | SMS-сообщение с напоминанием о платеже |
| `robot_call` | Автоматический голосовой звонок |
| `operator_call` | Звонок живого оператора контакт-центра |

Назначение типа воздействия выполняется **не случайным образом**, а на основе пропенсити-скора клиента, что создаёт реалистичный selection bias.


### 7.2 Этапы генерации синтетического uplift-слоя

| № | Переменная | Описание |
|---|---|---|
| 1 | `BASE_PD` | Базовая вероятность дефолта без коммуникации |
| 2 | `CONTACT_PROPENSITY` | Вероятность инициации контакта банком |
| 3 | `COMMUNICATION` | Тип воздействия (treatment) |
| 4 | `RISK_SEGMENT` | Сегмент риска клиента (low / medium / high) |
| 5 | `CONTACT_HISTORY` | История контактов (основа fatigue effect) |
| 6 | `PREFERRED_CHANNEL` | Предпочтительный канал коммуникации клиента |
| 7 | `INTERACTION_SCORE` | Взаимодействие характеристик клиента и воздействия |
| 8 | `DELAY_FLAG` | Флаг отложенной реакции клиента |
| 9 | `TRUE_UPLIFT` | Истинный каузальный эффект коммуникации |
| 10 | `TARGET_AFTER_CONTACT` | Наблюдаемый бинарный исход после воздействия |


### 7.3 Реализация функций генерации

Ниже приведены реализации каждого из десяти этапов. Все функции принимают датафрейм `df` и возвращают его с добавленными переменными.


In [ ]:
comm_types = ["none", "sms", "robot_call", "operator_call"]

#### `generate_base_pd(df)` — базовая вероятность дефолта

Вычисляет латентную вероятность дефолта клиента без учёта коммуникаций. Используется линейная комбинация наиболее информативных признаков: `EXT_SOURCE_1/2/3`, `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_INCOME_TOTAL`, `SK_DPD_MAX`, `CREDIT_DAY_OVERDUE_MAX`. Результат нормализуется в диапазон [0, 1] через сигмоид-функцию.


In [ ]:
def generate_base_pd(df):

    features = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "AMT_INCOME_TOTAL",
        "SK_DPD_MAX_x",
        "CREDIT_DAY_OVERDUE_MAX"
    ]

    X = df[features].copy()

    # 1. нормализацация признаков, чтобы масштаб признака не отражал его вклад
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 2. линейная комбинация
    weights = np.array([-1.5, -1.5, -1.5, 0.5, 0.5, -0.5, 1.0, 1.0])

    score = X_scaled @ weights

    # 3. добавляем шум
    score += np.random.normal(0, 0.5, len(df))

    # 4. sigmoid → probability
    base_pd = 1 / (1 + np.exp(-score))

    df["BASE_PD"] = base_pd

    return df

#### `generate_contact_propensity(df)` — пропенсити к контакту

Моделирует политику банка по инициации коммуникаций:

```
CONTACT_PROPENSITY = 0.6 * BASE_PD + 0.2 * (AMT_CREDIT / max) + 0.2 * (AMT_ANNUITY / max)
```

Клиенты с высоким риском и крупными кредитами получают более высокий пропенсити-скор. Это намеренно создаёт **selection bias**: treated-группа систематически отличается от контрольной по уровню риска.


In [ ]:
def generate_contact_propensity(df):
  '''Policy банка (propensity к контакту)'''
  p = (
        0.6 * df["BASE_PD"] +
        0.2 * (df["AMT_CREDIT"] / df["AMT_CREDIT"].max()) +
        0.2 * (df["AMT_ANNUITY"] / df["AMT_ANNUITY"].max())
    )

  df["CONTACT_PROPENSITY"] = np.clip(p, 0, 1)

  return df

#### `generate_treatment(df)` — назначение типа воздействия

Назначает тип коммуникации на основе пропенсити-скора через вероятностную модель:

| CONTACT_PROPENSITY | Вероятности [none, sms, robot, operator] |
|---|---|
| > 0.7 (высокий риск) | [0.10, 0.20, 0.30, 0.40] |
| 0.4 – 0.7 (средний риск) | [0.25, 0.35, 0.25, 0.15] |
| < 0.4 (низкий риск) | [0.55, 0.30, 0.10, 0.05] |

Механизм имитирует реальную стратегию контакт-центра: дорогие каналы направляются высокорисковым клиентам.


In [ ]:
def generate_treatment(df):
  '''Генерация типа коммуникации'''
  def assign(p):
      r = np.random.rand()

      if p > 0.7:
          probs = [0.1, 0.2, 0.3, 0.4]
      elif p > 0.4:
          probs = [0.25, 0.35, 0.25, 0.15]
      else:
          probs = [0.55, 0.3, 0.1, 0.05]

      return np.random.choice(
          ["none", "sms", "robot_call", "operator_call"],
          p=probs
      )

  df["COMMUNICATION"] = df["CONTACT_PROPENSITY"].apply(assign)

  return df

In [ ]:
def generate_segments(df):
  '''Сегментация клиентов (важно для uplift)'''

  df["RISK_SEGMENT"] = pd.qcut(
      df["BASE_PD"],
      q=3,
      labels=["low_risk", "medium_risk", "high_risk"]
  )

  return df

#### `generate_segments(df)` и `generate_contact_history(df)` — сегментация и история контактов

**Сегментация**: клиенты делятся на три группы по квантилям BASE_PD: `low_risk`, `medium_risk`, `high_risk`. Сегмент используется для моделирования гетерогенного treatment effect.

**История контактов**: количество предыдущих контактов моделируется из распределения Пуассона с параметром `λ = 0.5 + 3 × BASE_PD`. Более рисковые клиенты контактируются чаще. Значения ограничены диапазоном [0, 10].


In [ ]:
def generate_contact_history(df):
  '''Сегментация клиентов (важно для uplift)'''
  lam = (
      0.5 +
      3 * df["BASE_PD"]
  )

  df["CONTACT_HISTORY"] = np.random.poisson(lam)

  df["CONTACT_HISTORY"] = np.clip(df["CONTACT_HISTORY"], 0, 10)

  return df

#### `fatigue_multiplier(history)` — эффект насыщения контактами

Эффективность коммуникации снижается экспоненциально с ростом истории контактов:

```
fatigue = exp(-0.35 * CONTACT_HISTORY)
```

При большом количестве предыдущих контактов мультипликатор стремится к нулю, что соответствует феномену **communication fatigue**.


In [ ]:
def fatigue_multiplier(history):
  '''Функция fatigue - Эффект контакта уменьшается с ростом количества контактов.'''

  return np.exp(-0.35 * history)

In [ ]:
def generate_channel_preference(df):
  '''Channel preference - У клиента есть “любимый” канал коммуникации'''

  channels = ["sms", "robot_call", "operator_call"]

  df["PREFERRED_CHANNEL"] = np.random.choice(
      channels,
      size=len(df),
      p=[0.5, 0.3, 0.2]
  )

  return df

#### `generate_channel_preference(df)` и `generate_interactions(df)` — предпочтения и взаимодействия

**Channel preference**: каждому клиенту назначается предпочтительный канал коммуникации (SMS — 50%, robot_call — 30%, operator_call — 20%). Эффект коммуникации усиливается при совпадении назначенного и предпочтительного канала.

**Interaction score**: `INTERACTION_SCORE = income_norm × (1 − credit_norm)`. Клиенты с высоким доходом и низкой кредитной нагрузкой имеют более высокий score, что модифицирует величину treatment effect (гетерогенность эффекта).


In [ ]:
def generate_interactions(df):
  '''Treatment interaction (гетерогенность эффекта) - Некоторые признаки усиливают эффект коммуникации'''

  income_norm = df["AMT_INCOME_TOTAL"] / df["AMT_INCOME_TOTAL"].max()
  credit_norm = df["AMT_CREDIT"] / df["AMT_CREDIT"].max()

  df["INTERACTION_SCORE"] = (
      0.5 * income_norm +
      0.5 * (1 - credit_norm)
  )

  return df

#### `generate_delay(df)` — отложенная реакция клиента

Часть клиентов реагирует на коммуникацию с задержкой. Вероятность задержки: `delay_prob = 0.3 + 0.4 × BASE_PD`. При `DELAY_FLAG = 1` непосредственный эффект воздействия снижается вдвое.


In [ ]:
def generate_delay(df):
  '''Delayed response - Некоторые клиенты реагируют только через время'''

  delay_prob = (
      0.3 +
      0.4 * df["BASE_PD"]
  )

  df["DELAY_FLAG"] = np.random.binomial(1, delay_prob)

  return df

#### `generate_true_uplift(df)` и `generate_uplift_dataset(df)` — итоговая сборка

Функция `generate_true_uplift` вычисляет истинный uplift-эффект как произведение всех компонент:

```
effect = base_effect * fatigue * channel_bonus * interaction * delay_factor + noise
```

Функция `generate_uplift_dataset` последовательно вызывает все генераторы. Финальная вероятность дефолта: `PD_after = BASE_PD + TRUE_UPLIFT`, затем бинаризуется в `TARGET_AFTER_CONTACT`.


## 8. Итоговый синтетический датасет

**TRUE_UPLIFT** — истинный причинный эффект коммуникации на вероятность дефолта клиента. В реальных наблюдательных данных это значение ненаблюдаемо: для одного клиента невозможно одновременно зафиксировать исходы при наличии и при отсутствии коммуникации (проблема контрфактического вывода).

В синтетических данных TRUE_UPLIFT формируется как детерминированная функция от:

- сегмента риска и типа коммуникации (базовый эффект)
- истории контактов (fatigue multiplier)
- совпадения канала с предпочтением клиента (channel preference bonus)
- INTERACTION_SCORE (усиление/ослабление эффекта по характеристикам клиента)
- флага отложенной реакции (delay factor)
- случайного шума

Финальная вероятность дефолта: `PD_after = BASE_PD + TRUE_UPLIFT`.


In [ ]:
def generate_true_uplift(df):

    effect = np.zeros(len(df))

    low = df["RISK_SEGMENT"] == "low_risk"
    mid = df["RISK_SEGMENT"] == "medium_risk"
    high = df["RISK_SEGMENT"] == "high_risk"

    effect += np.where(
        (df["COMMUNICATION"] == "sms") & low,
        -0.05,
        0
    )

    effect += np.where(
        (df["COMMUNICATION"] == "robot_call") & mid,
        -0.06,
        0
    )

    effect += np.where(
        (df["COMMUNICATION"] == "operator_call") & high,
        -0.12,
        0
    )

    # негативный uplift
    effect += np.where(
        (df["COMMUNICATION"] == "sms") & high,
        0.03,
        0
    )

    # fatigue
    fatigue = np.exp(-0.35 * df["CONTACT_HISTORY"])
    effect *= fatigue

    # channel preference
    match = df["COMMUNICATION"] == df["PREFERRED_CHANNEL"]
    effect[match] *= 1.5

    # treatment interaction
    effect *= (0.7 + df["INTERACTION_SCORE"])

    # delayed effect
    delay_mask = df["DELAY_FLAG"] == 1
    effect[delay_mask] *= 0.5

    # saturation
    saturation = df["CONTACT_HISTORY"] > 5
    effect[saturation] += np.random.normal(0.02, 0.01, saturation.sum())

    # noise
    effect += np.random.normal(0, 0.01, len(df))

    df["TRUE_UPLIFT"] = effect

    return df

In [ ]:
def generate_uplift_dataset(df):

    new_cols = {}

    pd_features = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "AMT_INCOME_TOTAL",
        "SK_DPD_MAX_x",
        "CREDIT_DAY_OVERDUE_MAX"
    ]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[pd_features].copy())
    weights = np.array([-1.5, -1.5, -1.5, 0.5, 0.5, -0.5, 1.0, 1.0])
    score = X_scaled @ weights
    score += np.random.normal(0, 0.5, len(df))
    score = score / 3
    base_pd = 1 / (1 + np.exp(-score))
    df["BASE_PD"] = base_pd


    # CONTACT_PROPENSITY
    p = (
        0.6 * df["BASE_PD"] +
        0.2 * (df["AMT_CREDIT"] / df["AMT_CREDIT"].max()) +
        0.2 * (df["AMT_ANNUITY"] / df["AMT_ANNUITY"].max())
    )
    new_cols["CONTACT_PROPENSITY"] = np.clip(p, 0, 1)

    # RISK_SEGMENT
    new_cols["RISK_SEGMENT"] = pd.qcut(
        df["BASE_PD"],
        q=3,
        labels=["low_risk", "medium_risk", "high_risk"]
    )

    # CONTACT_HISTORY
    lam = 0.5 + 3 * df["BASE_PD"]
    history = np.random.poisson(lam)
    history = np.clip(history, 0, 10)
    new_cols["CONTACT_HISTORY"] = history

    # PREFERRED_CHANNEL
    new_cols["PREFERRED_CHANNEL"] = np.random.choice(
        ["sms", "robot_call", "operator_call"],
        size=len(df),
        p=[0.5, 0.3, 0.2]
    )

    # INTERACTION_SCORE
    income_norm = df["AMT_INCOME_TOTAL"] / df["AMT_INCOME_TOTAL"].max()
    credit_norm = df["AMT_CREDIT"] / df["AMT_CREDIT"].max()
    new_cols["INTERACTION_SCORE"] = (
        0.5 * income_norm +
        0.5 * (1 - credit_norm)
    )

    # DELAY_FLAG
    delay_prob = 0.3 + 0.4 * df["BASE_PD"]
    new_cols["DELAY_FLAG"] = np.random.binomial(1, delay_prob)

    # добавляем сразу
    df = pd.concat([df, pd.DataFrame(new_cols)], axis=1)

    return df

In [ ]:
def assign_communication(df):
    """
    Назначает тип коммуникации (treatment) на основе CONTACT_PROPENSITY
    и RISK_SEGMENT, имитируя реальную стратегию банка.
    """
    # 1. Определяем, будет ли вообще контакт с клиентом (Bernoulli trial)
    # Используем CONTACT_PROPENSITY как вероятность успеха
    is_contacted = np.random.binomial(1, df["CONTACT_PROPENSITY"])

    # 2. Создаем массив для типов коммуникации, по умолчанию "control" (нет контакта)
    comm_types = np.full(len(df), "control", dtype=object)

    # 3. Для тех, с кем контакт состоялся, выбираем тип согласно бизнес-логике [2]:
    # - Высокий риск -> звонок оператора
    # - Средний риск -> робот
    # - Низкий риск -> SMS

    high_mask = (is_contacted == 1) & (df["RISK_SEGMENT"] == "high_risk")
    mid_mask = (is_contacted == 1) & (df["RISK_SEGMENT"] == "medium_risk")
    low_mask = (is_contacted == 1) & (df["RISK_SEGMENT"] == "low_risk")

    comm_types[high_mask] = "operator_call"
    comm_types[mid_mask] = "robot_call"
    comm_types[low_mask] = "sms"

    df["COMMUNICATION"] = comm_types

    return df

In [ ]:
# Итоговая последовательность вызовов:
df_sim = generate_uplift_dataset(df_sim)   # Создаем латентные признаки и сегменты
df_sim = assign_communication(df_sim)      # Назначаем воздействие (Treatment) [2], [3]
df_sim = generate_true_uplift(df_sim)      # Рассчитываем эффект на основе назначенного воздействия [

/tmp/ipykernel_18694/3133509302.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["BASE_PD"] = base_pd


In [ ]:
df_sim = df_sim.copy()

In [ ]:
uplift_cols = [
    "BASE_PD", "CONTACT_PROPENSITY", "RISK_SEGMENT",
    "CONTACT_HISTORY", "PREFERRED_CHANNEL",
    "INTERACTION_SCORE", "DELAY_FLAG", "COMMUNICATION",
    "TRUE_UPLIFT"
]

# Добавляем в датасет, в котором пропуски не были обработаны, новые uplift-поля
df[uplift_cols] = df_sim[uplift_cols]

In [ ]:
df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,SK_DPD_MAX_CC,BASE_PD,CONTACT_PROPENSITY,RISK_SEGMENT,CONTACT_HISTORY,PREFERRED_CHANNEL,INTERACTION_SCORE,DELAY_FLAG,COMMUNICATION,TRUE_UPLIFT
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,NaN,0.950968,0.609805,high_risk,9,sms,0.450668,1,operator_call,0.028513
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,NaN,0.608977,0.456933,high_risk,0,operator_call,0.341462,1,control,-0.005344
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,NaN,0.227401,0.148340,low_risk,0,sms,0.483622,0,control,0.003340
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.0,0.359173,0.253956,low_risk,2,robot_call,0.461974,0,control,-0.021130
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,NaN,0.629868,0.420203,high_risk,2,robot_call,0.437186,0,control,-0.004420
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,NaN,0.661227,0.430674,high_risk,3,operator_call,0.469229,1,control,-0.001041
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,NaN,0.671782,0.425683,high_risk,2,sms,0.467030,0,operator_call,-0.068337
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,NaN,0.423995,0.311099,medium_risk,0,sms,0.416992,0,control,-0.010546
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,NaN,0.376648,0.259927,low_risk,1,sms,0.455039,1,control,-0.012077


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 161 entries, SK_ID_CURR to TRUE_UPLIFT
dtypes: category(1), float64(99), int64(43), object(18)
memory usage: 375.7+ MB


In [ ]:
# Сохраняем как CSV
df.to_csv(os.path.join(PROCESSED_PATH, "uplift-dataset.csv"), index=False)